In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import pickle
import os
import subprocess
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_1_run_cvae_ddp import *
from e_2_CVAE import *


# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs' # hes or bs
barr_type = 'van' # van or barr
opt_type = 'call' # call or put

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942

# training

In [4]:
import os, subprocess, sys

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"

code = """
import torch, os
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("device_count =", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
"""

subprocess.run([sys.executable, "-c", code], env=env, check=True)

CUDA_VISIBLE_DEVICES = 0,1,2,3,4,5,6,7
device_count = 7
0 NVIDIA GeForce GTX 1080 Ti
1 NVIDIA GeForce GTX 1080 Ti
2 NVIDIA GeForce GTX 1080 Ti
3 NVIDIA GeForce GTX 1080 Ti
4 NVIDIA GeForce GTX 1080 Ti
5 NVIDIA GeForce GTX 1080 Ti
6 NVIDIA GeForce GTX 1080 Ti


/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


CompletedProcess(args=['/home/ajoufe/anaconda3/bin/python', '-c', '\nimport torch, os\nprint("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))\nprint("device_count =", torch.cuda.device_count())\nfor i in range(torch.cuda.device_count()):\n    print(i, torch.cuda.get_device_name(i))\n'], returncode=0)

In [ ]:
# CVAE DDP training settings
dim_z       = 8 # 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = 5 # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 1e-3, 1e-4, 1e-5, 1e-6
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 30
resume_path = None # 이어서 학습하고 싶을 때
save_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size*7}_{bn_chunks}_{lr}_{beta}_{warmup_chunks}_chunk{num_chunks}.pt"

n_samples = 10000 # n_samples= 1k, 10k, 100k
if n_samples % 2 != 0:
    raise ValueError("n_samples should be an even number for antithetic sampling")

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

gpu_ids = "0,1,2,3,4,5,6" # 7 GPU problem
nproc = len(gpu_ids.split(","))

hidden_dims_arg = ",".join(map(str, hidden_dims))


In [4]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = gpu_ids
env["NCCL_DEBUG"] = "INFO"
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
env["NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc}",
    "f_run_cvae_ddp.py",
    "--model-type", model_type,
    "--dim-z", str(dim_z),
    "--hidden-dims", hidden_dims_arg,
    "--batch-size", str(batch_size),
    "--num-chunks", str(num_chunks),
    "--lr", str(lr),
    "--beta", str(beta),
    "--save-path", str(save_path),
    "--num-workers", "8",
    "--prefetch-factor", "2",
]

if warmup_chunks is not None:
    cmd += ["--warmup-chunks", str(warmup_chunks)]

if use_bn:
    if bn_chunks is not None:
        cmd += ["--bn-chunks", str(bn_chunks)]
    cmd += ["--use-bn"]
if resume_path is not None:
    cmd += ["--resume-path", str(resume_path)]

time1 = time.time()
subprocess.run(cmd, env=env, check=True)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWar

eta min/max 계산 중
eta min/max 계산 중
eta min/max 계산 중
eta min/max 계산 중
eta min/max 계산 중
eta min/max 계산 중
eta min/max 계산 중
계산 완료

계산 완료

계산 완료

계산 완료

계산 완료

계산 완료

계산 완료

ajoufe1:17337:17337 [0] NCCL INFO Bootstrap : Using enp13s0f0:192.168.0.15<0>
ajoufe1:17337:17337 [0] NCCL INFO NET/Plugin : No plugin found (libnccl-net.so), using internal implementation
ajoufe1:17337:17337 [0] NCCL INFO cudaDriverVersion 12050
NCCL version 2.14.3+cuda11.7
ajoufe1:17342:17342 [5] NCCL INFO cudaDriverVersion 12050
ajoufe1:17340:17340 [3] NCCL INFO cudaDriverVersion 12050
ajoufe1:17343:17343 [6] NCCL INFO cudaDriverVersion 12050
ajoufe1:17341:17341 [4] NCCL INFO cudaDriverVersion 12050
ajoufe1:17339:17339 [2] NCCL INFO cudaDriverVersion 12050
ajoufe1:17338:17338 [1] NCCL INFO cudaDriverVersion 12050
ajoufe1:17337:17415 [0] NCCL INFO NCCL_IB_DISABLE set by environment to 1.
ajoufe1:17337:17415 [0] NCCL INFO NET/Socket : Using [0]enp13s0f0:192.168.0.15<0>
ajoufe1:17337:17415 [0] NCCL INFO Using network Soc

[W reducer.cpp:1298] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later iterations to have unused parameters. (function operator())
[W reducer.cpp:1298] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later 

[18:35:33] [rank 3] epoch=1 chunk_pos=0 done
[18:35:33] [rank 1] epoch=1 chunk_pos=0 done[18:35:33] [rank 2] epoch=1 chunk_pos=0 done

Chunk step     1 | epoch    1 chunk   1/100 | file_idx  48 | BN train  | beta_eff: 1.0000 | Recon: 1.9176 | KL: 0.1028 | Total: 2.0204[18:35:33] [rank 4] epoch=1 chunk_pos=0 done
[18:35:33] [rank 6] epoch=1 chunk_pos=0 done

[18:35:33] [rank 5] epoch=1 chunk_pos=0 done
[18:35:33] [rank 0] epoch=1 chunk_pos=0 done
[18:35:34] [rank 4] epoch=1 chunk_pos=1/99 ci=8 load start
[18:35:34] [rank 3] epoch=1 chunk_pos=1/99 ci=8 load start
[18:35:34] [rank 5] epoch=1 chunk_pos=1/99 ci=8 load start
[18:35:34] [rank 2] epoch=1 chunk_pos=1/99 ci=8 load start
[18:35:34] [rank 0] epoch=1 chunk_pos=1/99 ci=8 load start
[18:35:34] [rank 1] epoch=1 chunk_pos=1/99 ci=8 load start
[18:35:34] [rank 6] epoch=1 chunk_pos=1/99 ci=8 load start
[18:35:52] [rank 3] epoch=1 chunk_pos=1 dataset loaded len=67108864
[18:35:52] [rank 3] epoch=1 chunk_pos=1 dataloader ready batches=2340

In [ ]:
num_chunks = 5
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size*7}_{bn_chunks}_{lr}_{beta}_{warmup_chunks}_chunk.pt"
save_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size*7}_{bn_chunks}_{lr}_{beta}_{warmup_chunks}_chunk{num_chunks}.pt"

In [ ]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = gpu_ids
env["NCCL_DEBUG"] = "INFO"
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
env["NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc}",
    "f_run_cvae_ddp.py",
    "--model-type", model_type,
    "--dim-z", str(dim_z),
    "--hidden-dims", hidden_dims_arg,
    "--batch-size", str(batch_size),
    "--num-chunks", str(num_chunks),
    "--lr", str(lr),
    "--beta", str(beta),
    "--save-path", str(save_path),
    "--num-workers", "8",
    "--prefetch-factor", "2",
]

if warmup_chunks is not None:
    cmd += ["--warmup-chunks", str(warmup_chunks)]

if use_bn:
    if bn_chunks is not None:
        cmd += ["--bn-chunks", str(bn_chunks)]
    cmd += ["--use-bn"]
if resume_path is not None:
    cmd += ["--resume-path", str(resume_path)]

time1 = time.time()
subprocess.run(cmd, env=env, check=True)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")


# use BN

In [ ]:
use_bn = True
bn_chunks = 5 # None or num
num_chunks = 5
resume_path = None
save_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size*7}_{bn_chunks}_{lr}_{beta}_{warmup_chunks}_chunk{num_chunks}.pt"

In [ ]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = gpu_ids
env["NCCL_DEBUG"] = "INFO"
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
env["NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc}",
    "f_run_cvae_ddp.py",
    "--model-type", model_type,
    "--dim-z", str(dim_z),
    "--hidden-dims", hidden_dims_arg,
    "--batch-size", str(batch_size),
    "--num-chunks", str(num_chunks),
    "--lr", str(lr),
    "--beta", str(beta),
    "--save-path", str(save_path),
    "--num-workers", "8",
    "--prefetch-factor", "2",
]

if warmup_chunks is not None:
    cmd += ["--warmup-chunks", str(warmup_chunks)]

if use_bn:
    if bn_chunks is not None:
        cmd += ["--bn-chunks", str(bn_chunks)]
    cmd += ["--use-bn"]
if resume_path is not None:
    cmd += ["--resume-path", str(resume_path)]

time1 = time.time()
subprocess.run(cmd, env=env, check=True)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")
